In [ ]:
## imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path

MODELS_DIR = Path("/content/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## Data Downloading and Transformation

In [ ]:
# 1. Define the transformation
# VAEs often work well with pixel values scaled between 0 and 1, which ToTensor() handles automatically.
transform = transforms.Compose([
    transforms.ToTensor() 
])

# 2. Download and load the training dataset
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', 
    train=True,
    download=True, 
    transform=transform
)

# 3. Create the DataLoader
batch_size = 16
train_loader = torch.utils.data.DataLoader(
    train_dataset, 
    batch_size=batch_size,
    shuffle=True, 
    num_workers=2
)

# 4. Define the human-readable class names (Fashion-MNIST has 10 classes)
fashion_classes = (
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
)

print(f"Total training images: {len(train_dataset)}")

### Regularized MLP - VAE 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    x = x.view(x.size(0), -1)
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl, recon_loss, kl

In [ ]:
class VAE_MLP_Regularized(nn.Module):
    def __init__(self, latent_dim=32, dropout=0.1):
        super().__init__()
        
        # Encoder
        self.fc1 = nn.Linear(784, 512)
        self.ln1 = nn.LayerNorm(512)
        self.fc2 = nn.Linear(512, 256)
        self.ln2 = nn.LayerNorm(256)
        
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        
        self.dropout = nn.Dropout(dropout)
        
        # Decoder
        self.fc3 = nn.Linear(latent_dim, 256)
        self.ln3 = nn.LayerNorm(256)
        self.fc4 = nn.Linear(256, 512)
        self.ln4 = nn.LayerNorm(512)
        self.fc5 = nn.Linear(512, 784)

    def encode(self, x):
        x = x.view(x.size(0), -1)
        h = F.relu(self.ln1(self.fc1(x)))
        h = self.dropout(h)
        h = F.relu(self.ln2(self.fc2(h)))
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z):
        h = F.relu(self.ln3(self.fc3(z)))
        h = self.dropout(h)
        h = F.relu(self.ln4(self.fc4(h)))
        return torch.sigmoid(self.fc5(h))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

In [ ]:
def train(model, train_loader, epochs=10, lr=1e-3, beta=1.0):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0
        total_kl = 0
        total_recon = 0

        for x, _ in train_loader:
            x = x.to(device)

            recon, mu, logvar = model(x)
            loss, recon_loss, kl = vae_loss(recon, x, mu, logvar, beta)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step(
            total_loss += loss.item()
            total_kl += kl.item()
            total_recon += recon_loss.item()

        print(f"Epoch {epoch+1}: Loss={total_loss:.2f}, Recon={total_recon:.2f}, KL={total_kl:.2f}")
    # save model
    save_path = MODELS_DIR / "vae_mlp_regularized.pth"
    torch.save(model.state_dict(), str(save_path))
    print(f"Model saved to: {save_path}")

In [ ]:
train(model=VAE_MLP_Regularized(), train_loader=train_loader, epochs=1, lr=1e-3, beta=1.0)

## CNN MODEL - Baseline & Deep

In [ ]:
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    total = recon_loss + beta * kl
    return total, recon_loss, kl

In [ ]:
class VAE_CNN_Baseline(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()

        # Encoder: 1x28x28 -> 64x7x7
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),   # 32x14x14
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # 64x7x7
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(64 * 7 * 7, latent_dim)

        # Decoder
        self.fc_dec = nn.Linear(latent_dim, 64 * 7 * 7)

        self.dec = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), # 32x14x14
            nn.ReLU(),

            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),  # 1x28x28
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.enc(x)
        h = h.view(x.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z):
        h = self.fc_dec(z)
        h = h.view(z.size(0), 64, 7, 7)
        return self.dec(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

In [ ]:
class VAE_CNN_Deep(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()

        # Encoder: deeper stack
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=1, padding=1),   # 32x28x28
            nn.ReLU(),

            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # 64x14x14
            nn.ReLU(),

            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 128x7x7
            nn.ReLU(),

            nn.Conv2d(128, 128, 3, stride=1, padding=1), # 128x7x7
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(128 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(128 * 7 * 7, latent_dim)

        # Decoder
        self.fc_dec = nn.Linear(latent_dim, 128 * 7 * 7)

        self.dec = nn.Sequential(
            nn.ConvTranspose2d(128, 128, 3, stride=1, padding=1),   # 128x7x7
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),    # 64x14x14
            nn.ReLU(),

            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),     # 32x28x28
            nn.ReLU(),

            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.enc(x)
        h = h.view(x.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z):
        h = self.fc_dec(z)
        h = h.view(z.size(0), 128, 7, 7)
        return self.dec(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

In [ ]:
def train(model, train_loader, save_path, epochs=50, lr=1e-3, beta=1.0):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()

        total_loss = 0
        total_rec = 0
        total_kl = 0

        for x, _ in train_loader:
            x = x.to(device)

            recon, mu, logvar = model(x)

            loss, rec, kl = vae_loss(recon, x, mu, logvar, beta)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_rec += rec.item()
            total_kl += kl.item()

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss: {total_loss:.2f} | "
            f"Recon: {total_rec:.2f} | "
            f"KL: {total_kl:.2f}"
        )
    # save model
    MODELS_DIR.mkdir(exist_ok=True)
    full_path = MODELS_DIR / save_path
    torch.save(model.state_dict(), str(full_path))
    print(f"Model saved to: {full_path}")

In [ ]:
model = VAE_CNN_Baseline(latent_dim=32)
train(model, train_loader, save_path="vae_cnn_baseline.pth", epochs=1, lr=1e-3, beta=1.0)

In [ ]:
model = VAE_CNN_Deep(latent_dim=32)
train(model, train_loader, save_path="vae_cnn_deep.pth", epochs=1, lr=1e-3, beta=1.0)